# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [ ]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json


True

In [3]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [21]:
import os
load_dotenv(override=True)
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
deepseek_base_url = os.getenv('DEEPSEEK_BASE_URL')
deepseek_model_flash = os.getenv('DEEPSEEK_FLASH_MODEL')
deepseek_model_pro = os.getenv('DEEPSEEK_PRO')
openai = OpenAI(base_url=deepseek_base_url, api_key=deepseek_api_key)

In [16]:
# Some lists!

checklist = []
completed = []

In [17]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [18]:
get_checklist_report()

''

In [19]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [9]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [12]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [11]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [22]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [15]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [26]:
def loop(messages):
    response = openai.chat.completions.create(model=deepseek_model_pro, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model=deepseek_model_pro, messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [24]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
Quiero que analizes las opurtonidades de negocio en el mercado actual para un Inginiero de IA que vive en Smolyan, Bulgaria. No tienen porque ser todos de tipo local.
Incluye oportunidades tanto locales como internacionales, y considera diferentes industrias donde la IA pueda tener un impacto significativo. Pero con clara prioridad local.
Usa solo el castellano como idioma de output.
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [27]:
checklist, completed = [], []
loop(messages)

Checklist #1: Enmarcar el perfil del ingeniero de IA y el contexto de Smolyan (Bulgaria): coste de vida, ecosistema
tecnológico, fiscalidad, idioma y ventajas UE
Checklist #2: Identificar oportunidades locales y regionales en Bulgaria con prioridad en Smolyan y alrededores
Checklist #3: Identificar oportunidades internacionales y remotas compatibles con residir en Smolyan
Checklist #4: Analizar industrias concretas de alto impacto para la IA y su encaje local/internacional
Checklist #5: Evaluar y priorizar las oportunidades con criterios de viabilidad, ingresos e impacto local
Checklist #6: Estructurar recomendaciones finales y un plan de acción por fases

Smolyan: ciudad media en las montañas Ródope, economía basada en turismo (Pamporovo), agroganadería, forestal, 
administración y pymes; no es hub tecnológico. Bulgaria: UE, impuesto plano del 10 %, coste de vida bajo, internet 
aceptable en núcleos urbanos, huso EET. El perfil: ingeniero de IA con capacidad de trabajar remoto para clientes 
UE/globales y de digitalizar pymes locales.

Checklist #1: Enmarcar el perfil del ingeniero de IA y el contexto de Smolyan (Bulgaria): coste de vida, ecosistema
tecnológico, fiscalidad, idioma y ventajas UE
Checklist #2: Identificar oportunidades locales y regionales en Bulgaria con prioridad en Smolyan y alrededores
Checklist #3: Identificar oportunidades internacionales y remotas compatibles con residir en Smolyan
Checklist #4: Analizar industrias concretas de alto impacto para la IA y su encaje local/internacional
Checklist #5: Evaluar y priorizar las oportunidades con criterios de viabilidad, ingresos e impacto local
Checklist #6: Estructurar recomendaciones finales y un plan de acción por fases

Oportunidades locales: IA para turismo de montaña (Pamporovo, Chepelare, alojamientos rurales), agroganadería y 
forestal (visión artificial, predicción, drones), administración local (smart city, atención ciudadana, fondos UE),
salud y telemedicina de montaña, educación/formación, pymes y comercio, y eficiencia energética con 
biomasa/renovables. Modelo realista: boutique de IA local + integración con hubs de Plovdiv/Sofía.

Checklist #1: Enmarcar el perfil del ingeniero de IA y el contexto de Smolyan (Bulgaria): coste de vida, ecosistema
tecnológico, fiscalidad, idioma y ventajas UE
Checklist #2: Identificar oportunidades locales y regionales en Bulgaria con prioridad en Smolyan y alrededores
Checklist #3: Identificar oportunidades internacionales y remotas compatibles con residir en Smolyan
Checklist #4: Analizar industrias concretas de alto impacto para la IA y su encaje local/internacional
Checklist #5: Evaluar y priorizar las oportunidades con criterios de viabilidad, ingresos e impacto local
Checklist #6: Estructurar recomendaciones finales y un plan de acción por fases

Oportunidades internacionales/remotas: consultoría y freelance en IA/ML para clientes de la UE, UK y EE. UU.; 
empleo remoto asalariado; micro-SaaS vertical global (turismo rural, e-commerce, legaltech, healthtech, agritech); 
APIs y modelos; startup con sede en Bulgaria para aprovechar el impuesto del 10 % y programas europeos; educación 
online y asesoría en cumplimiento del AI Act.

... (4 outputs left out)

Checklist #1: Enmarcar el perfil del ingeniero de IA y el contexto de Smolyan (Bulgaria): coste de vida, ecosistema
tecnológico, fiscalidad, idioma y ventajas UE
Checklist #2: Identificar oportunidades locales y regionales en Bulgaria con prioridad en Smolyan y alrededores
Checklist #3: Identificar oportunidades internacionales y remotas compatibles con residir en Smolyan
Checklist #4: Analizar industrias concretas de alto impacto para la IA y su encaje local/internacional
Checklist #5: Evaluar y priorizar las oportunidades con criterios de viabilidad, ingresos e impacto local
Checklist #6: Estructurar recomendaciones finales y un plan de acción por fases

Plan de acción por fases: 0) preparación legal/técnica y búlgaro básico o socio local; 1) caja inmediata con 
consultoría remota internacional y pilotos en turismo local; 2) producto turístico empaquetado y extensión 
regional; 3) proyectos agro/forestales, administración y micro-SaaS global; 4) salud, energía/clima y financiación 
europea. Incluye construcción de ecosistema (meetups, universidad, asociaciones) y ventaja fiscal del 10 %.

Checklist #1: Enmarcar el perfil del ingeniero de IA y el contexto de Smolyan (Bulgaria): coste de vida, ecosistema
tecnológico, fiscalidad, idioma y ventajas UE
Checklist #2: Identificar oportunidades locales y regionales en Bulgaria con prioridad en Smolyan y alrededores
Checklist #3: Identificar oportunidades internacionales y remotas compatibles con residir en Smolyan
Checklist #4: Analizar industrias concretas de alto impacto para la IA y su encaje local/internacional
Checklist #5: Evaluar y priorizar las oportunidades con criterios de viabilidad, ingresos e impacto local
Checklist #6: Estructurar recomendaciones finales y un plan de acción por fases

Oportunidades de negocio para un ingeniero de IA en Smolyan (Bulgaria)

Lectura del contexto: Smolyan no es un polo tecnológico, y precisamente ahí está la oportunidad. Hay poca 
competencia local en IA, coste de vida bajo, impuesto plano del 10 %, pertenencia a la UE y una economía real de 
montaña —turismo, agroganadería, forestal, pymes y sector público— con necesidades digitales poco atendidas.

---

PRIORIDAD LOCAL: OPORTUNIDADES CERCANAS

1. Turismo y hostelería inteligente — prioridad máxima
- Zona: Pamporovo, Chepelare, lagos de Smolyan, agroturismo y casas rurales.
- Productos: chatbot multilingüe 24/7 en búlgaro, inglés, griego, alemán y rumano; análisis de reseñas de 
Booking/TripAdvisor/Google; previsión de ocupación y precios dinámicos; generación de contenidos de marketing.
- Modelo: suscripción mensual más implementación. Empezar con 3–5 pilotos para crear casos de referencia.
- Estimación orientativa: 150–500 €/mes por alojamiento y 3.000–10.000 € por proyecto inicial.

2. Digitalización de pymes, comercio y administración local
- Automatización documental, RAG sobre normativa búlgara/europea, atención ciudadana, CRM con IA, análisis de datos
municipales.
- Financiación: licitaciones locales, Plan de Recuperación y Resiliencia de Bulgaria y fondos europeos de 
digitalización.
- Impacto rápido: menos papeleo y mejor servicio en comercios, gestorías, inmobiliarias y ayuntamientos.

3. Agroganadería y forestal de precisión
- Visión artificial para control de ganado, monitoreo de cultivos y bosques, detección de plagas, control de 
calidad de madera y uso de drones.
- Socios naturales: cooperativas, asociaciones forestales, universidad local y programas PAC/UE.
- Proyectos subvencionados estimados: 20.000–150.000 € según convocatoria y consorcio.

4. Salud y telemedicina de montaña
- Zonas rurales envejecidas y dispersas: triaje, automatización de informes, recordatorios, monitoreo remoto de 
personas mayores.
- Empezar por lo no clínico para evitar la regulación sanitaria europea; después crear alianzas con clínicas y el 
hospital regional.

5. Educación y formación en IA
- Cursos, bootcamps y reciclaje profesional para hosteleros, pymes y jóvenes de la región.
- Estratégico: crea cantera local, reputación y futuros clientes.

6. Energía y clima
- Optimización de calefacción de biomasa, eficiencia energética en edificios, microgeneración renovable, predicción
de riesgo de incendios forestales y adaptación climática.

---

OPORTUNIDADES INTERNACIONALES Y REMOTAS

Estas no exigen salir de Smolyan y sirven como motor de caja mientras madura el negocio local.

- Consultoría freelance en IA: IA generativa, LLM/RAG, MLOps y visión artificial para clientes de la UE, Reino 
Unido y EE. UU. Estimación: 50–100 €/hora o 5.000–15.000 € por proyecto.
- Micro-SaaS vertical global: convertir el aprendizaje del nicho local en un producto escalable, por ejemplo “IA 
para turismo rural y de montaña” vendido por suscripción a toda Europa.
- Empleo remoto asalariado: ingeniero de IA/ML para empresas tecnológicas europeas o estadounidenses.
- Startup con sede en Bulgaria: aprovechar el impuesto de sociedades del 10 % y programas como Horizon Europe, EIT 
Digital o EIC Accelerator.
- Asesoría en cumplimiento IA/GDPR: ayudar a pymes europeas a prepararse para el AI Act y la protección de datos.

---

INDUSTRIAS DE MAYOR IMPACTO

- Turismo y hostelería — encaje local fuerte y rápida monetización.
- Agroganadería y forestal — impacto estructural en los Ródope y fondos europeos disponibles.
- Salud — alto impacto social, pero regulación y ciclos más largos.
- Administración pública — alta demanda, contratos estables y financiación UE.
- Educación — impacto medio, clave para retener talento local.
- Energía y clima — alto impacto a medio-largo plazo.
- Manufactura — más regional, en torno a Plovdiv.
- Fintech, insurtech y ciberseguridad — principalmente internacional/remoto.

---

PRIORIZACIÓN RECOMENDADA

Carril A — Generar caja ya: consul

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>